# Лабораторная работа №1 — единый расчёт стандартизованной модели

**Для локального запуска:** `python -m pip install numpy pandas scipy matplotlib scikit-learn statsmodels openpyxl ipython jupyter`

**Выполнить:** `Kernel → Restart & Run All`. Одна ячейка запуска всех этапов не требуется: команда **Run All** последовательно выполняет весь ноутбук.

**Исходная модель:** восемь регрессоров стандартизированы на объединённой обучающей выборке (1365 CPU). Целевая переменная остаётся `y = ln(CPU Mark)`. Оценивание коэффициентов и система нормальных уравнений выполняются **непосредственно по стандартизованным регрессорам**. Никакие альтернативные спецификации модели не подбираются.

**Обозначения:** $\beta$ — неизвестные теоретические коэффициенты, $B=(b_0,\ldots,b_8)^T$ — МНК-оценки, $Y$ — вектор наблюдений, $\hat Y=XB$ — вектор прогнозов, $X$ — матрица с единичным столбцом и восемью стандартизованными факторами. СНУ: $X^TXB=X^TY$.

Один запуск сохраняет графики PNG (без диагностического анализа остатков), четыре Excel-файла для матриц (в схеме лекции 2, слайды 12–13), CSV с численными результатами и ZIP всех результатов. **Никакие листинги кода не предназначены для вставки в отчёт.**

В ноутбук встроен точный снимок очищенного набора `final5.csv` (1665 наблюдений), поэтому скачивать исходный CSV отдельно необязательно.

**Excel:** файлы генерируются только через общедоступную библиотеку `openpyxl`; закрытый `artifact_tool` не требуется. Все Excel-файлы сохраняются в `lab1_pumba_standardized_results/excel/`.


In [1]:
from pathlib import Path
import io, base64, zlib, zipfile, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
from IPython.display import display
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

DATA_OVERRIDE = None                 # если нужно использовать иной CSV, вписать путь
DISPLAY_FIGURES = False              # True — показывать все графики в ноутбуке
RESULTS_DIR = Path.cwd() / 'lab1_pumba_standardized_results'
FIG = RESULTS_DIR / 'figures'
TAB = RESULTS_DIR / 'tables'
APP = RESULTS_DIR / 'appendix'
EXCEL = RESULTS_DIR / 'excel'
for folder in (FIG, TAB, APP, EXCEL):
    folder.mkdir(parents=True, exist_ok=True)
# Удаление файлов предыдущих версий: эти диагностические графики больше не строятся.
for retired in ('05_residuals_numeric.png', '06_residuals_dummy.png',
                '07_residuals_diagnostics.png'):
    (FIG / retired).unlink(missing_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','axes.titlesize':10,
                     'axes.labelsize':9,'xtick.labelsize':8,'ytick.labelsize':8,
                     'figure.dpi':105})
pd.set_option('display.max_columns',30)
pd.set_option('display.float_format',lambda v:f'{v:.6f}')
print('Сохранение результатов:',RESULTS_DIR.resolve())

Сохранение результатов: /home/monkofwst/codes/smmo/pravki_final/lab1_pumba_standardized_results


## 1. Встроенные данные и воспроизводимое разбиение

In [2]:
# Сжатый неизменяемый снимок final5.csv, встроенный в файл ноутбука.
_EMBEDDED_CSV_PARTS = (
    'eNqFfevOLbet5P88ywdDJCVKeoQBBphHMHw8nuTgJHFgOxicefrplrpbrFKvzz+MOHsvr9UXipdisfjzr7/98vvXf/z0+y8//vz3X3/+rx//+r',
    'f/9/XH//7Xj//36+/6488//fy3X378x398/d3Wv//8r3//+I+ffvuvr//+5afffv/xp7/++vWPn/757//z089//Pu3X3778X/8849f/o5/9L/+',
    '+Nsvv339/q9ffv79x3/85++//+c///rjz7/++59/fP380x+//PXX3/77x//507/++PVff5Ev+aF+1fRD+ko/aPk6/0V6+ZL6dfzJl3ylv+Qv/S',
    'F/5Xr+zfGPH/+U4v4ldv71+aG/6PGR9GVyfeT8P57Dl8j4Hf3q42fk/h3RLynjAzq+RH44vvL8G7u+qJifXyLPl6Qf2pe2H3z8bfqhfHm9P2DX',
    'V/iXlesaxuUm+orzfk1/KOM/P39Ms80rlec78vEH1w+MS9GSzytdt2vHl5QcPmKlHj8i10fS+Ih8eQk3I2K2LiWNH2rzYs9vOS+41eOvMjxWn1',
    '9yP9ZWfH83ft/wuOnajm9Iz0fOK/F1sef/mkv/Oh/+c7HnY8vH395vMPe0PiDHz5yXKvfPnCYguefzI/cLzsfP2FcvwUpyOg3J4WeO22/xIy1P',
    'E5hfcry7w0qOx309tvMj1qye35KubznfYCFL0uN+L0uycS16XIuBxVrq8Y6ut3y/m2GO6Xz0Fi5Xz5u+73d8QhMa0/n0Sz8+dhuTVL2NSa9b1n',
    'nL43iN30tJ0ZqG3d+PJQ+Dq309uflwy1fL10fOj1Ypilc7/+PyXLH6+MD9bM8P9K8cX1Dp4emn+7HBJ1ICs/ZhTJLHrZ6/VaZhNz1NTp5XpIfx',
    '623Z85CBO9D9p9zL/ZpluZQSzlhrEh/KPD/l/sQ4Ib3d5+f+Dl2/Mi66qUXTP++50FNJx8Pv4dnrYbU92v5hbi2eQh+vZz4Vv96PaO/2fE267n',
    'ic1Oc1zm+R8Pg7niBz78tUpu23r9bDgx0nSJ5LkekB23PLp9ncv6LXddT1XMe19Frjg718ZLqt/ryQZfi3P/Dr5d9PrR6W38BFHj+dwickW70v',
    '9baSNB/9fTOW0v1c7bFpj08+t3W3t4PU+25PI9Gmjgf9tIHHh47nbhqfuw1rvZ779MMinu8Lua1Ep0G3+27q6UMFjtd1Hm5v3ktfZ+cOgSV42X',
    'Eo/Ho109KOg3391/MblMOszCd2P5DaenSg528IRkjxxCHy+EOJ4a3Y5kraioDTRJJEnyUrdt0na4SMHA6Wxec+TCTHV3MYwBUbx7k5j+F4wFJL',
    '7tGbXK4aHVdB13b6Yavh2R/G5vHh63gw/Xpy43orxC8dp69avOlyXq+HWxqPLl1HOA9rS+Wxerhpvf10zflwAyl8R183M6xJqgV/Q05r3Eya1x',
    'qfiM4LudMcyzVGJhveBm06l4I2bcPCorupp7dHd2Prd8azLRp/Z6Y52mJy0SS6z2kpmoKlWGnL06frWvN9reNwmUn8CIWL8eQKBYPT34RPnAZy',
    'xlm/LH/7knn3KVp1nq9dNDyTmrqDGz/fH7ydXgsfDedMNxfOdDkRyhCsZT2Tx9FLjTGHLmRky9niu5lJg0PgsmR3bnhnqVdQepxB7znagC5Xf3',
    '9EpaXolmR8S6k/WFnvOGdOqm3lUcPWGsZhO45wj0ag3lNMT2bkkgKJ7PmTCfIkpSRUj9MV/NI8Xng/mlr0FfOc2x2qp0F65UT1OjxPnmpVDleQ',
    '8LnJVWhMR1cM/SyUEeOGquXgT3S4RbjY8TPhli93UaO7KKezVjgbGQ3hOGAen62MXNf7+YCvN6ipxEw2j0/06LjUa8P8/rxai5XT4S4yFhKQbr',
    'UZFQoHDiG/1HKPPyQrHD9Fq7bH4OR6h/M437G0nbkFPrdCeUHXleRM79e5VJT1TO64gxbZDue2vJ9FJ3v5tnw+Eqxn9KtiPVMr+gvwXOfdFNeY',
    'T+cVAUNC1jACjo+A4ScpWInEBGSc0r7V+Yo3nLVVyPvOOPuY/bC3maHGT5RVS46j1p78U3arz9PqdX97cDOip5HgQa9ojZbEuAj0mejcEeE46M',
    'pnJ2HKJZoU/dKIkvEEujWs40dSnkO8zhUqSR8ulOqqPhJVJbPPkHSJYlDQYUslxAQ4f1c0hvxQMuaHW/rhRRHKGZVZPBcXshHr7zTP5/PQ6hkC',
    '5a0AXJGnFCxDTmfe/ImBhznVHE/oLEM8gh8jNCUqZZ6APiKTaqwzZ46DT6QDqDQBB2khFkt9ooFeYUdWeT7CTj1LUaO8YRjJHQWrOobr/WbO0C',
    'VvmaG+o2wTsqiAaqhDPG8zYD116HRJ59vpIXY9L+fOlluWmC3L476fR1LySmAerKcGUEPzeW7wVOhALO5TrqWTjx5VhsQ81x3zz7MiQjOS1uPP',
    '5HElEHKk5RzL2XmxNfxMaVvY6lTMpFQYLSorbM3EsMaEO9QxjwM2ET6eR5Twb9DYy5Bica5nKowBNF0g2kr8euZr0ZWB2kDICifUcvxhfLKtYf',
    '2GAOaMwhUD5JmCwpfo8DcKzy0vWzsvtuMJvTJdicljU3x/GpGgca3uiV9PxjfYqzP6dUQyjQlO9Q3ADMXbfCbn7wgYJCHhuVdwn/OhSLvRIpvO',
    '8owqDRCHTqiTntWKwwWXdZInMNWGKayPPMaiq5YUBsDSCsd5OhV/vMHmqueXnHlBfoE27jrxKL64TuQUVL1wGK3r4aYBbjiW6BYP+8gLxNtLcl',
    'Hjl+yYesb41tw5vmU067xwKXmiaEZ80yJyNaMKxsiasCLSu357jsaA/43CG9Yh2XW9YGGL1HnwqSaS8bDyYwK1Zy53DFJlV2Hvp2hoh4+t0dBs',
    'HB6s4XO3+yM3FF4Y5m6AKvkEHu9vOWvoal1DQQSZ/4UZ9oTx/PzEgEifSqZl9KBPQ2QhEkJPfj8UHs6nXJUztIC01BSx8nk+W4o23/QBufXxoN',
    'ft3Mn/qIwx+edUNxskF3IFvh9W/Z39Qcj0gfYR5jYzjpNGGKf0EPXnkzv8rMT03yzHmwZQ965minAEk6dTkac1be2mgo9fRklklD/maPhVK+fC',
    'bVrtnVQd+XXoD9zuMYNhu3gs9F+evuYzo8oMl8NZl8boR0fkaWQgDkcss6MWjLYe4kZ74kY3NYAdzqQZQ2GXyqmoPlnoiEJZGQptK2rPpmKlju',
    'HpiqE3Jilzo7VgXq0CeXVwK8/5KL1zmpIwRcwnEJMo8mg/veAdeprEhkcePQIw/Zo8f6gm74+03tnPGuZLrhlhv60qUi0xGZpwBCAorePpyavp',
    '9TwSx2woL8juuZ0qBb9lWFiC/C9LdDyzoUs9/bThBYYNjeOGGiDVw9YguxtZsXIRXoLTt7O9RvfsdClVE6bFo74avvI+8PiSCVQY/URpmEU+Df',
    'mVgQgkVFfzpF+WPf5poUOzgWAje8vYogv5+d2VzLJ1Rgo1CcRqqL8gsb7LgGrzjMpjkIItgBzbiffxyojvW0W8W8e1pphCZhNMy0YdkKMZnOn3',
    'islvvQZ1KH1enIFo32EHwHy0c54yul/RUHw0DJW6ANCOyMdbBpsdhY+FuDHaEVhDKzmDVPkIZurGp0iIeTAhDdaWG3zJ7Do6PrXk6P2eLCScr+',
    '8O6fiIS/+A+y1o9wyVSrUrFIxXZyTespJLubpjsRxJxBEZ9ZO+PdpVSZgwKBDAtIkOptgltfh+7nYRn3W9E+f1VMpWsfjtru88xaB4vZAhjSBH',
    'a+jRH3cw/ckowaphV/+lVd7YJLeqshiAquObG3yi5h2npNqIfTHiAhOiUCYy+GprTHBI+3Hco3OySEoaD9gqEpsgBZ/J5tbPywzf1swkBKb5DG',
    '9coIRq1P0/SRdb919j/JHewWG/0dusVIa9G9LKWg/u645zDpadtaLbfw78k2zWlrgE9pVljNzXG9eml8UtmwxY5B3ZkRGWtbJf4Y+k0hjpmGDk',
    'czvluNbGjideqlcos9rqhS+gsZzFaYMrIa9/PHpqxo1MM6WYNuUSew4T4VM/q60bazS/D5C+d4M6FMke2QrlAoy1jqiMtWeLVZapx6bTRDkKFg',
    'unN7Y3V/vgogPhy29o1lPTj8a8UQrRBUKhGBIQJeIL456bI8IHLDmdOH5nPPJ4sjWCjd4i9HYB7CmA9LL89XrHcHjqoqTo9fBz4EreDIAaLMVi',
    '5nV5fV+oDFEuF4J+mAFcy+MlH4+RRSLbJFAdnw5lz9H0px2YRAwCu3FEoJnuTTO+5JFDID9KHvDuvpLLJttN7ZPsd/9esd+9GFKeuWsuiHfVmv',
    'arVYg/PdWNAgc1Rx4NcbRroWpBtDH/yZ9XMy3FECQEUDoNT9y5yCJCnxmAd28N17bYequ7oZGa21pjtFLJkjTVvUsNSbyfR6MTZyyyuY63lzk4',
    'NXysMoqjDE50MaTSxaOxwX9iIJ5Al/T4fH1wAbQ3kxIb0W+5ZDluqX1vtCenoQCraGOq5opo1cQY49WaVcxDQs56u+N0HqH2yeJmb+EEMoSSIi',
    'oHBktDEMZDNkjzD3yE0XifiWKOD3c+f7wjAr0upC8mk2bGsJguEG867C0Nydj/q824dsnrhsftnMkmUWQahY4mGuJcXh5DARVzoBtQT81NG9cU',
    'xLaSAZBy2YEY3sJ/BE/ZY025K8T+0yQVChMvGluRM5PE0yENXiDBo+N3LMff8Qje3BZ5VNxG4LQgO1e7vLxjaLu5M6k57WB8eqPiP/Gp9gLe+m',
    'l4Pommlspw1vnzvTwNXp+VdII5CWxxlIzk3J2lobVwxV5Xgj5StyMqEF3k4XTpXRJWRTQYjCBPbgthSE+R9WDXdpKghGrPpygZxubkIjfmwxYp',
    'gTttTyMEy5a+uNPjY8VitvOKmvlbY/W74jSvJ3on2Cln5vtXap5PQDIyDjL1l9Qqw9+UPbg3zh7KgwKvciMRMkoDGzkMbKTrbrSG1Nrdp8VKQP',
    'OhvtXRhHTmtsS+zhijiCypva+Qk3NfofLITeKRG1/xYF5J7pyJGr1iVeFiXDnwKJQKoSZZafGRMgkkGYE2eRMevX/IVB5fXbEoCRzdUGyX5xXd',
    'XjSPFu/TF2hgTTKRaZm/NCvtyZAWBYr0KLeJotkYlkyUfnnpDCg6s3YSdhcs8jGGn/RWmQpzPbxQoZ7Uj07IJZC7zo4kuadMP3ThUTEwkDPNRk',
    'Y1J7BiwlmdaNS+KrHZYu/ccGlIdRrjMIXa1t4ufJvIX7cL7IszN316nIraEMeL8S1EYO/MxmjKBlfPo1pWc2hhSPo4DQXUMuILD/kyEmFK3dyo',
    'E/JZK9Mm0yq1B+cjmeKoy0MceSDyVDaI3Gdb9Kkq3SUGqLaANb2RwqMaSImGB4wwVk8FMVaLcNakz9YGWTHQteePndaYmPUIzOQT3CFSCPdkWj',
    'KO7Uot8n7UYg14I0KUt7a1KEJfdKRdZYxLIAdXL/LCuCOtTACrWw21EUsyPpITXqVCjMmIKsJgSaJAuCcHioT82jNPBtiy2NnV6cvJSpwwuf3A',
    'ERTSB3wijGIWHkXqX37DMWO6IDPPl/OYjnnMS/Wq6QVxxtGPVLYCVwj6qYUtgIgWpaRKRAueMHFt5G+oLPJe+JhTC/ioETZwoo1n9jgkzYuU/I',
    'ymWooVQLuIoC/tzgcFSWoU/pRHU72hhzWefDxCbd6hfGCfHImosYvV08XmwErY2OuNKrQmjTvJPNQ7qnkm/SOxzlIcSWxhZrRNnsoYnzvyix6a',
    'iBaH8PxhaTqAAk4zF2nVpcFdQFqwRshtvEZdU7B3HXcc9gIA38BKIJIajfWm81sCL2E+fqHC5gEJh1Em5ZDf8IQldEszTGuKxBI3rAJGyMhgTS',
    '3xrFjHvvcgHhdyXBHjLU33Y4y92SowGDDzl3gztWb+wMQwnwItTcCAaOUAqxpFfDsTizi6mE2chmzbGuK7+l2KuZisyaqHYisI5zyd83UlgE9Z',
    'jAbXdyh2TF8NLZ2JYfvWP3rGuYAxg44D8TkJGr1uXMMmkX90td8pv29hxuRp3wIDSawzXqPkMCQLj3Elpq02ghmfoYvnHVlvHJ/IM6mkyrldnn',
    'jac8YW/0GfiE1W2zZ+uRFbtDWezUdSvhdDOOYZdV8kv6TMmjPuS5TCBUJG5Mhw3O/KmL/xbEQrkcliamXjlbiHAqA75/9EUTfpfcdUiwAfxJzw',
    'w4rJY00jMyS71/BESto4uEKYwVBwwO6vbOyIwtSHPq9kjQAxuKTx5Uyws3fGxRkDkcb5RWMezagBM7zA9O1k9kXbgikSr8a0krLOzvCzSbh5qD',
    'QifpYiBlQOx4DtZ0HTqW6ih3/PC66Hn/mE9gSA6SOb8DguVecH66tQnMeVZxlGFhOBoTofyZ0yjyFyi+nFhchk7t2m2btd0CyE0le05cgB8+e5',
    'mDlVuA3gHJlmDioPpzoGp7yC/KI8MLVMhQI4tzCTpCtdPU/y4/MVsjuuOC/PROOlIA0zzseQHEgUUfmGCs8Z+5MZXzGk1A5jt1c9Hwu5vD0WKu',
    'TMqzKDwvmYCdPVRrEXm/QnpMNjt0rU7ZThnFkc5rmr6CYcxho6fm+d6YRKJ744k94qEzFWffthxA3EVj5MK0/GYfRPNHVbiyZoo2yktzETBvzw',
    'WABdFGTrJV6tIuKS79EJA8y5o8ZFTpxs8lyEaUOO+XgpET2S1vh4EPKTNb1Mh2KzMzmrH2SC/qrgI9l4w3k0FeS73znii+1tMByqTVqIIUZTLd',
    'KQyvkGOaeuXMwJDbFpdrT7vXteceoosNVW6embtlJjml9pWA7o3bFeYdc7j30RieK4I0HhgdAnuWvc/pL5CnhSRfLJM2i1EmxLsaetK2IuyEUq',
    'jjvuRKbTsPtbrv/AQ0h/eNFUMen73BFyzL0zZxFG8soJysQjGrWKloJTGs1opea6AstcnUmw5x2nB87UJ8NeVMArbY2DYWXD7zYGmL4wzZDM2V',
    '/6/NixPluzf0JgHYDWR1KiPvMI35c3o9vlQPtoNGmTxQjK50wwExl6Q4AsuVI+c5xPsOtIqrodD7BculWOuFs4HT6jAODMRUVPjsPCU2wF1IJ8',
    'Iy50Rpqq8xRApxB1PP13mG+9HhyQvyjvwBCHgYQgfxTyg+zwLR7RwklBPR5S4gKmcWGBEN12mk8i8zaz0GlUtBUGPOyrwRB10Ep4qclHEtwd+a',
    '3bwIiPIoiDFM4ud6CMehwmmM+kVyv0SDpXuVlehCThmA6NLAcdkmeK5mkW97zzsouE81Mrmz6n6bpx/C+DXPNL2JsQHOSVBUfdd/xhkj7VjQRG',
    '8IDojtajfxt2YEx7gziXK8a5h9P2dBi1Mye+L+B5mKz2+oGPttJR3/QmbcqFrleYPM55ZoyWechaSoHZ5p21WOpWcxd6QUO8AyNHSCGu5NmZbu',
    'N4wgrSBd/0A5I17oRcE40Pyf/wTJVLIOWR++IFj9Ci3z+ZRskabzsUbcukXJ3FGRo2RI732DkvcpxgHuzgAhT9Sh7ZUt1lKf3+ifEaURdxf4tW',
    'a2SwUi5/tZN95406QKCts0yqPspt4z2q7S1Aiv4i+/AXEhc1GUv8Uf1ZBowqpIsHvtZLQwL4+SU4Oen+ku1TLDTd4U+82ovdEIOu0DD1SI6RDi',
    'NfgCFojR2tYNhr+rVVLBxvbaKF2vdNx4VnlAYLLHF3NRKRct9q/7ZJbGYutzfJNOE2RkLVQs2lMTi29Xmr7a1RnCmx5SbXZCt3EAzVpyxy3vPI',
    'ZryQPBjfT9EdMATCR/FYueR95nE2IZSkDFAvVIVJb6R/WrVyWtzXuLzOfknZiRb6gSy4bA06/eXiq70IgaxKGYe2SBBhpprlzq31Q7dYyjYyV9',
    'BdtFKY3WrLHHUiZ9sIDQnCjNnKDI/NKV3taRO2LDhkc54unDQQakTlzloGZQn+TBEdyy8zD8jpMf8wPLYILCWxpG9i5n3ek1Vq1Net39KJnJKT',
    'MK2kYI6oSTPniEf+1+BkQGY9kRsbleVT9ecPKeKK+91emJzxjkshhbFTjhyEDDo2IByg0VvWq6fEYGNmrc5cmO5mk/31TNnUzEHQcIjQRByfvs',
    'XpS5v6ib5DRFiFnYSb/mma1G9Jchxs5UmdUozb9EeszVGkeOtDJRqJLAlm9t8gWsFs6rIlGhiG9o+h7OR05U0+lMCLPFIrNLzOt4wzD1o6V6aV',
    'K1PP36h7TaHiygCRszKs76wDVO5tXVFjxVatd/elrZRKIqdtDUk/QoBO94NofZfMCGtaI5GjTDOAomQOfrGiRa1DODu9V7ij52WGweW8JWpAeF',
    'Hh1pmTXurS4w8US4hi1VawlHe51CoMGusXILUdKF4TTASdYXXrXNM7/Yzmwj/jVBGq7ZojL0RA1kT2GE3P4X+u6bHXqwVNf9R3wPAfds0CU6hD',
    'UIow4Zf4WbnXTaKjcUHYt6mtQnKquW2KSgZaSKfhUwmcaGqy950RYIFcN6RuMwVTqjWy7XLiWLFUbZjNwtj4xR5xfiSkzGtOYgcWj84M/d2oA/',
    'g66YbW2Mkas1Mz5eEkrmZK21rxJDuvWbfU20k+dmAL+vYttxX0/kYojDSWnpXzTCp9bUzC4TSwcanePXFwcQazrMVR7AuUiSl8YK/qh6an1a0L',
    'xeqTvSC0vWk7Za+C7MZNA8Ja5tEI9AV6vuT2HQb1UFRWryWfdV7MuuwF8cRpEuE6gWd4cyn44KfWXKhJh668QCZakSdmtQrxxLZjbi9LOQAntr',
    'JJegoCE6VwTpa+hZ8yToAO+uso4+VdGPuhfPsHDDiw3owHZ2kCYdgZInudWDsee5mPoUESI4kEgh+Z0zWUWQt/pGDlazUjQ2UrrrKXwo0YLrDP',
    '2cEGrDie1jcq0rchxuJiSMQYADxI0K5SI4h6wtKirDxlxfDU6Os59efRDYhnZhYbAslDLdA/jWpdU46FQ5OQKx/D0YVqUsW40wUcAWgCTOZdYe',
    'adbHT6l3Y1RiZzoRy00JTC+JLAj02zlzkfXHomjqRY2UciaqCwuPs3+FMJMi4JljU5bpYyVj3ZgL2me9YGUy9L+1ygonzSd8/twzqT2yJ96Qgv',
    'h1ErJFNeOV77cm4X1EJC0S8tB2N3P2oNC9oQaS1ceNVnOdIPJdpJEIqeN+yZB/LYrMXqNxNuc4igGkMtgsS5kjZpAaHeurWdL4+n57TZ90ToMW',
    'uplXvVmWcDk+3OjSpOlw8o8nMtxAT0+FSull3viQ5Gxipbhxg/0rapLasv/REubYeiqrxP+d5IV+74Q898dVtPTnFZxcssat6XzMGGnWRRO+oN',
    'IzbcsXPZAfb9Ok+ENa7lEVCbSZsBQtVW/+V+aNR7V+mMINLmohqvZFuVkB5EJ1M9KtAGKOUFRo5bbexU7+0kZOrwM4PWVd7wgHU7nnAzFXT4B5',
    '7WK0uVXLqTa5WFJ0aa5XnDc25w0xAh1O6i5WEBh8Q92absMnfzvDo3YDpqF5SaWH/UsEAore5bbYCiUXv7k22FgeZpOCe5NLgbH4p+rvYIaXnK',
    'DJc6S1cSmGAbS9py+aBG/eQwyYVHknhoV5yXUzG5rPNMGchEX6L/QAeyNfLyQIOpZcbvC6Hzo59kb8nS4x2TdibHFjTpuvhY8okwJA1Xr4zQBC',
    'fUt0mG05/Goramztl7w/6K97aRPLY6vtO+p8H6iqwiRbbcy3LMRhI9kY89k7JJyK4tw7yXRhhydkqT7N17CKaaV1i/sWZnrNm3TukGpLSXGQCk',
    'bMneOSSaTWntg7bhkyDWupNSc9Q/8s4LAEvo/l+tNlgSeEET2FFo2RyT+HxGjhQZpa36ewH8OEFvLBFSv1oIPnkfnim0QbOX+mHpRYj721xYgc',
    'H3AYTZJ23ey5a2ZXT5BP7qaicQ7nAmVLjbzU1Y8I5lLDKvZWPYKEndF/yhYGT2XbUDKStp22AllL2r266fRNuPotuBGLcaersOUFnIrU99BMpD',
    'N2F4ax/2ET7BdqncC64SzPupWCzEjgypvB3hwDC0h5hptFoWAZ3Uv+n9z7aH2s5FId6ybtQMblJXA96FxQykBFUv/3Z/wAKG7INKRq3G5HPSLM',
    'qGqM+bMY22SMgPZ1i43PWTHR9ZcU7kL5yb8+a8nUgIF7C8Z5pw1pN+mBe/f6Vb5Z1AlxL0M5qriTfuysJL53cofseuR5N5vMn4NY/NbB1o1Imq',
    'qzwAqPU80pjvDc5An5xZPxBac/KdJk8zF9fMM7UWoWdLc3C38l5cz+HcOiECwJHeJSAt7yt917iLvffp5u0oJeeQEnsCXXKLWoTX4REtO86lsM',
    'B2bD70LQcJwFCduh1eSEEuYeUzsodGnBScISUdDA0TukF2su7ib1FcQNZ+54V4cpq4rTPgQZIxF+Of9ASHlUveUSxMUhYpWT6sziaa6MscSfbu',
    '+woidDxDjThTX6PU5yQfb2ajRwehizTFQa7GsGAl8Hwg2y5i5jwn5JA/Fl5b0oTnXYI29hSlzHG+V4YYAPH5lTcdMMxVG62wHbl0/6yF93K+Sl',
    'Ou9CruQL3WxSDs0LFM14prWrZdg1WcW0U0+Ta3LQkFOFLvoTb4eHkRnteMw0gPiyfKH3TeoZNo768iXjobHzDyHCZQ9Fl9eeSlMbffz16hWYqa',
    'uNXjtE2ybbuj0tMtGidiVx5zbH317NhreGYxw9aNtl8HgtDeQeB26jKzpGj2LAyY+lrbO/GNbQEtyer0VDg99FUh+OQ+4X6qx78+9wyKoxYXut',
    '6yIN73sR3YhFDsZbcsTUDktI+54hayrrvoP8ioHZGAJJeBYHW12/N+KY9jHHX6Jfv7QbVKb3ZpBpRktRHk3r0tlDYgl3lIFKdvRT6rbxWtb7zq',
    'jaVQiVTjHVezsVKKvmw1NFgl03kubqoJx8SyJgZIeL6op7xzt5Fv2509MHWNc+rbRqeNp1z6Pv6FwTGnbTGp43iKJMssf6BncPzotS7qFI/xDa',
    'wzvVU0z4hRFZZvnG7pvtwgGqdPIxwyoLMv0j4VxhfltrAGTeVRztT2ZZC0YGdT1jzuGmifWjhjoHtx5a3Nu9C40AIlILRfDATft0zj8RryqfJp',
    'LGsy4TqT1tK2eyUZQ4P6FdnbgtuprhVjsLPQvHBD9/z2vB5K6zwZmah7doLqjVM1mKE1lqPaVJTrJ6Nfc/lVuYfQkFpaGhRdJH8xMqa6KSdsyq',
    'mVdlJuFAXR3rgHlGnLXG340JhINrwJriwqvBO9ZtySO8jB1GZYFKIlROAIcMomNlquGvHOLrNTn0g3JXnVdjfX7IldSGc6JxaoH8XYRymO/TUY',
    'SxmXkz02vqZn84iRqStD2awtXTXvQBuuzL44oxJ2H+Fg/iBuCM9BFwRSfV8W77yVqBXenphJuXOvEFnRoOs2Dk+7mnKSxCDncymPhpN4TKpfKp',
    'G+DRazCrwir+3a8QcsygZl86KKPXfcq7AoUkdZzVJKZvSqMYG4NYbj0lotMeW+bIlJbCj2aKBl59xBvt3h7ri58tLpOr+mQSu9sPZfHXyIQq1J',
    'j/C+bkhnuXe3XC4wwKX6HFScPFGcxKWeb5t6BRt/mIf7uujeR9XTWX7YbSRYB47QUBI3UqmH5j3/GVoq3nf2AJhKrqmg5Q+rvokmWwt79jtkCb',
    'DmGy3tUPS0i2keJitbNppk7+g1qncWUgmt/7n1QTZMXGmritM2DViRMHfv9fxhoGpVx/7NipFrYSS0Vj7IG6tut1yhlNjUjUkh15N8mqpb5lba',
    'Tg8wkh6xTeiVetDOBsmrX9cwlH5ytEEaf5vgHJVgNdZIFgJzWv4g2rMaivqy7AE4uyP2FxLPxyE0yTyyStztLEMvmEYIaZ/4i5gODeUjh+Rize',
    'P4eam8DVdIJVn5GL8I11rhCTJWVZTCGwUqgijdtrGHS+J9odC+C3DB+t9Hv13iTokOy2y0cwlODUVJFZeKQgkw++3FPizWfpKump+kS/Z0aSJP',
    'vvcqaLOIbfmSrr29eYqF5fU794MFskrDncjX4vmMGyedd3EoydP1fY8g790beAC3DKPDGWtHP25sG044J+6eUVXUMisHVcrachTSk/dypNeXoj',
    'UmF5a/A9vmlXRK7DaesqdudIpJZcpr2fcLY+vFW2y92GofLDJFLrvarOyCSwEtv8RGp1VPbHiUTyVXagXmsLJjDnqmTls9MnZ9CzecdvktR8XY',
    'S+ETktnKjrjQzj1pxqxpmlp1u0YRJe6cSpDmNKT7cann8qLXjHr3NXLe7xlc5E+JWIQxbenMr1ZEEcSWQLlxQJC+45iVOara+JnwUJelwhrjNC',
    'bV+5+sZPHU+j63gFkOyJq8thq8C4MkSn2kMQmAvjoNRmBss7a3duGz7UuEbZH0w3sp3FghwTnNCVvbj3b//fbUeY5+O+g1byuTEgklyNadyxiJ',
    'reWtxcMA5NhYsS2nhXWiSco+5UaiNLl+mOhaE3lNGZdtqCs0heCwnEmMl+5FeGXlmiLMPUirW5HmjCE//W3pQdnUo4hQUUZKLbQ6Hfj53fWDus',
    'BCl1J/2bsIT+XMUVgfF3dI+yY0T8sGrkkqGmCCtoj4bkxEe4vM+m0+bSShxe1PZBvu/RgxiTGiVOeMo7ej96UgfWesrGKUFnh6mZiMo7ctZ9YP',
    'LVRrpuY8iRpIu2W+4cx5kHGvvlVUfH5Z7Fe2DbhBymLkuqkzJ0MorLS65amV0E7pzkMLhV9Pq8wS2uSY0gszm1YSNFqoPKRGYN6qOM9bscTbOQ',
    'LK4gJYAKjByPg+fAQkyrs7bSTWmLkNYAzneGv+YZXDA2Uu+YcbMg2BNE9JGuc2AJNDejdW8DCaBw8bw14mqOej1cxHLFNxtUKpIXZ+g5VFAJp9',
    'cejNN8UZX1vNr/UWbV/2hQKUcWPsnX/AjkJLnnhGxpbkzJwi1G3fKAmrjb5wIawGgbKkjecnqKQpA3Vg94YbFT1tOMu+6Lj0BJJnm2xG9kZKy8',
    'ZE18NukYRn214saXnXZ0NTaRUSpgsKe7oB02M2MaZGFRpkcl+bx+QNfZg1B2NH2CfLjedwlZpttRiBZXVFhrlp0jttVOFxHFg0KWxyc9oQ6bBv',
    'I3FjS9qW8NR4llvnKppALCmCPR/YGD87ZQtWlee0o6mk4pwodm7WVPmT1uGfdbBq2gYtTpaBBdUuZWrN0CAN9CpX5l1nrEY07YlMDy3ZCREL0u',
    'I3CWwt23xZYyXNxIUEixJOqh9OhKDwV0GZ2Ut4MtIocVvH2+Cy4TzW2wBF2ni9iXZvues+Qo34hlWaqyTlm1KbfxzwfET+Cwtt9lWPkIzSC3m7',
    'PI19hqdQ/sO9c5sgUTJbnae6xr7Sx96tFl52SaSZWjYURWFyTLTwb/Bg0dgwmUh/DJlvpzDsNspN2yHXXMsK50rdscQNtMKSDmkjNfFq3PMjvN',
    'tYo0agnQTJ9om/P0le9k0zdhpSVFKn1OM2pFp4vW5B+EOTZH4omTpfRdPe6MZtBUWFhJiM6dK5d04/WE664+4mmc9222RZTimf/qnyuRuHPHBP',
    'gyc6lPEcFHQU05Q+MKpEPhK1Uxoq4wW150ANtH3IHfvqPelLEIwb+7ynF1VcFEVzhk2Zo4FrrF/QJVwBMD+A23xHx08gevlg9i6ZEClMcKPbLU',
    '03OQRb9eDcmt7Lvj0L13gJxfxRYCFLddNFSV/UuG95LV+6fT6qk6fsTK8vTA+FMuGVKzkr4PSdUohsC2S4YNSs+1jJcylz46n20Fa0CHFcd2ya',
    'udHGLU5ra+Xv6/CrSuPh10ojxZIZ9i7I02mNZ/9oNUYemlDlTVTtgfmHKkOhMXnDIMccuBfEu2f2pGxKzZjwbLQxXRub41nFBFo9seQu5iBWv3',
    '0s8jDmB+C2osycdt7/WUuHCmH4JFA6OLcXA3nG5gVGw6+tsVig07aPbjyUigliaS+ZO0lF123TfKUrsZRgD8TVC0fyrhVuWmhc8VnOvganGQS+',
    'elfGB+uRvkSJRX9RSadlYLr3RmhODfUtPXJD5wRTrWa0kJbrN0kbZppZ31yNacSsEFx0O4RCOWLNmzzH2HgZSCBm2JJV3tKhQ4DeOYMQsEgxXt',
    'R7CjOQBPcRkqlWaKhUZgPkNVYZSCgSwQ3+TI33Afzkt8Avq6vuVPyOEj7sVayTiRgdbboE657wk2oMP4r6e8OpZ+cJiEYNrISDMr60ywO5trmC',
    'KJMsYmUtl4T27fTtQ0raZR/GwIWjTXkfc1jwpLM+2ie/II3pCjqOjhFszPedVO/2qXs16XgpI10hja2Yyxn0hrtnBgNU9BleGfLnrhEPkKVp+R',
    'S4rFQw0jbQoHrTp5+pQdjV4i+iq7ihRhvD9IkClDUOULzOxbLuMrQoWaxddwFM2gGVfZ91wy7mQILT2wsKRF8Ue2m3+NFwXHdZZkNsttHOUPTq',
    'tbx4dY3jPSUza4GzwJYCZ/UVUPNtHJT9pCpJaQ4dHYEsX8pWJKF0dW5bwcG8yiTcnKjEEh06VPqG7DxhrKU4ahY8xlq0cgayTiGIWfJmPERUsY',
    'LSUl4wZxLCzJkvt6HcB3fO50Qv6Zd0TIpeoKrCc1V1zbLLPQfm0HynkltalPR+lCRQAWAJlOuHSc9k/AqdVBJddNumi4f5BAk7tTKrobjGtn6G',
    'u0sZl4q8KUYPAUscq+lMvKyVRcxle4GdwTfBFnxpmaufzpuXUuIuYiapFY3a1InHLKdbSK2x2AcvAh9SYUb4OW3YKA1Xk+2UAZ4IHIAjDD7F3W',
    'UwxPV0irvuRCc0ydS2KTwhZS3XTfjR+Ok31vcnEdLzE+1NInbpWtS2mwFqfRTPuG13VB2HP69LurMXHAq1SBSdGewdEuxpRIIQVUpxtHhSFWH1',
    'gnagZB3BZ0wXUPDRnhuv01XcEy4l7aLFyHfzrAzmJ5TyyM4Kscejy1fdOA2p7zwoQmUMUs28+rdB17jyYh9W89bODM6MUNYAkNCklQxpR5Qpi/',
    'SaOYvMqKPYFqUnCPyiery3fc8OIQNlPpG75z0qFlGA9wBGfOMTFPN4yq+19igkrCwkTCe0lNz3iQtEpiU1RlgrjRL1AvCpXONGA6vyR7NdOzDp',
    '5W64xdWn6UVzLD5/Ndk3n+J6Gq0bhrTtyXVmIFVeMdzzvg4Wc4PWK1Qt+4D/oiXYh12VWowjixNbUWt50eq7RvUeHQEHP9xXuL1EOYHxaLOs1D',
    'nXNIz28qM1QwX7LHpYu2Nmkwnr0+mX1vhvuj+izyvqQdRFWuEWBC8YzrXsE9E8DbGptjdeVGTb8p+gPzjSyVpiGfXCLWk5731GmOSSvsmPk4y2',
    'uGyzNzzyMlrNjNWjbnvatKYKs5N12zNgs93xSIrUyAN8u9ZB/c+Ui7YoZe+Jgd5NSlYr6QcFDV59eiW51lhBvUnyyVsHCEYIZvKsIBonq1l82X',
    '9jppIy2pQS7/wwSjd7zi8AG2oZbYIPLEaatzjWNsX7xmwzHospqNonSMSck0DGq2fLJgyTeciK5WXM6ocl97cxTEoBr/UVGJ5pOc6KG27OjEO+',
    '68mOEIO0+lZ4tkK4i3dt4sCpZkF5Ct/7ozxyVAnA6W/UOD4hlkFb4js7sGeiMtP8mvR31ESfPkaPLzkokNwiJczvKmkfz4BJg3IyF9t363BK4X',
    'mhXfeqF8xGNydpkndBJNoiaQ13EIda+wLXFCa9LxtI0QZqD8vr5W3Rro9EMdGCE7SS9lI0Ur2HcyAXTRbyQHXhhiXXR95SzJ5p7c7EOrzYZ7Gp',
    'YY+DgpR4J1XEAmG9s0SOWFD59I+7Y54dq2WjjfKC1KtjElMr3lckadeSwp5+cJLhniUyt2st94SGfliPJa1x6BBm1zGTmYeWdSxMTt8rnAlzO5',
    'jAqimzxTHsYrLDLtO73VVuW8n8YrPjjHxvH9jDaxtAxYTIIvg5F5LX+iKJgoqIJdPeeZ5NKopZxstGAc/7RiQET8V2yi5UuENo99subWlp61gW',
    'Ygar7eaoEQmptTZyKZ0oo0Ne0MExFex9exfeV5V5LMkTa3I0HlBrmQfUiHE9h2pZ9h7bkbUp65BXrFy0KFcuFQEkK4kNNlMOP1b/bQMl4LmEpU',
    'KfVbz3U8k4uRmo3frYyUY+IdnMLNuMYuJ+cth4GWZI4XcKIaPbFgba60csr5n2Lr7Gy2Ylf5ayOIk209Ve7ydW0k7qZDllZpDzXr+R6QtVCyh5',
    'adaZgsIiyC6V+QmXltriJyiLb7bvBX+vuqTiDpL6zRKZiVToxubrTDqTwmRczpxLYgUzWqDqRgBDvqWnlqLvMyhlDwCLavJ1mzzSNQw+V/VW6v',
    'i/aW5so7WKU+etKiNapFViOcHI1ntzu2ZlImxHtf+2IvILyXyKRmkHGAL3xV9oh+zXS8x60w+yKM80Tu2FT3xBYq/XlGnbOAMMyTKzvDJ274pW',
    'ZksPKbvZZ293d6iTZNf3+0O2SQ/xrIhMw0gW7euVtQkUVEZ0Z+iBxXaNtKq3DQmL97saAjUWPa6chjIzawld2+LB9Ajvpc7wHpcJ19xL+kZuTU',
    'tjaeir5ffo/Ag3Nwqw2M0K2xhvT8tIhgqT/E9p5X3xGmASfKXLti1o1VlK3ghVdeGOgVGwXsm9Lcn1Grd+F62so9Ypnreyk9QxyRGurm3D8bXa',
    'rtyIAGLtde8GYL15tZ6+nWhJnXUzjdaFWefBv8aD+uWlBxaLjJKFF2U3YhJmZ0hIEbDOtmESDBbUl14cEqomUy2FOeJC+pCk2H1hqgYbyouxTq',
    'HStu0xe61EjkDp9iYxll+ixSg90Ru78E6zdpoaazHQIJFaqvuyHFQCfiGmaPwOGRIlzhsbeaRP+4tSNk1FJE49qLXkupFfOeB4YmydeerW/WXS',
    'G1hOWljfp6DJugiKh4yOK3bni4F4qkZC8NU6FK6/qXVokYEB0uBLAq0qp3S059DNeTd45wQn60axaSClUVkwmPrY1hQ3VgwgFJowY7udEeMb9c',
    '+sCdNN7YRYcvDCwuaoBPlZ1H6R17EJ087bW2kicGh0FVIdsAQK8sZShkg38X3tmX+1AOpak11SWBvMDfjmPIWqobohfploisnyZkSkO6eZCx1W',
    'e7ZN7VlQD65IWXpw8t5KFeX+AK2TG3sVnGjnsJz81F0KBmBRqLLcG34bsr3S0NUK35FlF4zA1kplEfWXtTS57gsdMe9QoaXho3mZYtFmCZeSAP',
    'NpUOAd0/nL5IFm4omtpDw3PPOsvC82QcUkqY3prhnlIkKN8wYX6j0+h0VdYJkMBk3dJt/Y0krm3dkFG2Npa6jcr+bhUx5lrPPyrobAl7SyJq5f',
    'hTgGJ9O/34nsaaecI6dWNN0p/xIBwOmglF/eH1TuDVmmGefjJqO5MVzIQ3h1W/S9DUguifq0r9Gc4an69yoceu5c7RzFIyE2V2dCrHBHDLezvQ',
    'ipHJW9Qnd7lAIgAFCskt5cpxExka3+yCjhewRgf1nXWXlbsb154LUXu3HQItGRVgr3K1myOIMaj0/leEqyijemL9JccdW9WYIjA6NoU0C0yrZu',
    'ru/zzXiIu8ZD/P8B1iNjIw==',
)
if DATA_OVERRIDE is None:
    raw = pd.read_csv(io.BytesIO(zlib.decompress(base64.b64decode(''.join(_EMBEDDED_CSV_PARTS)))))
    data_origin = 'встроенный final5.csv'
else:
    raw = pd.read_csv(DATA_OVERRIDE)
    data_origin = str(DATA_OVERRIDE)
raw.index.name = 'row_id'
required = ['cores','base_clock_ghz','tdp_w','l3_cache_mb','cpu_mark','years_ago',
            'manufacturer_Intel','manufacturer_Other','category_Laptop']
assert len(raw)==1665, f'Для воспроизведения отчёта ожидается 1665 CPU, получено {len(raw)}'
assert raw[required].notna().all().all()
assert (raw.cpu_mark > 0).all()
assert not raw.duplicated().any()
print(f'Загружено {len(raw)} наблюдений; источник: {data_origin}')

Загружено 1665 наблюдений; источник: встроенный final5.csv


In [3]:
# Исторически то же разбиение, что использовалось в lab1_pumba.docx:
# train=1092, validation=273, test=300; после разработки модели
# финальное оценивание на объединённой выборке development=1365.
raw_features = ['cores','base_clock_ghz','tdp_w','l3_cache_mb','years_ago',
                'manufacturer_Intel','manufacturer_Other','category_Laptop']
features = ['log1p_cores','base_clock_ghz','log1p_tdp','log1p_l3',
            'age_centered','manufacturer_Intel','manufacturer_Other','category_Laptop']
feature_labels = ['Число ядер','Базовая частота','TDP','Кэш L3',
                  'Возраст','Intel (относительно AMD)','Other (относительно AMD)',
                  'Laptop (относительно Desktop)']

raw_development, raw_test = train_test_split(
    raw, test_size=300, random_state=42, stratify=raw.category_Laptop)
raw_train, raw_val = train_test_split(
    raw_development, test_size=.20, random_state=42,
    stratify=raw_development.category_Laptop)
raw_train,raw_val,raw_development,raw_test = [v.sort_index().copy()
    for v in (raw_train,raw_val,raw_development,raw_test)]
assert (len(raw_train),len(raw_val),len(raw_development),len(raw_test)) == (1092,273,1365,300)

age_center=float(raw_development.years_ago.mean())
def transform_features(frame):
    df=frame.copy()
    df['ln_cpu_mark']=np.log(df.cpu_mark.astype(float))
    df['log1p_cores']=np.log1p(df.cores)
    df['log1p_tdp']=np.log1p(df.tdp_w)
    df['log1p_l3']=np.log1p(df.l3_cache_mb)
    df['age_centered']=df.years_ago-age_center
    return df
train,val,development,test=[transform_features(df) for df in
    (raw_train,raw_val,raw_development,raw_test)]

# ОБЯЗАТЕЛЬНЫЙ ЭТАП: стандартизация ВСЕХ ВОСЬМИ признаков,
# включая 0/1 dummy-переменные. Параметры считаются на development,
# на test только применяется уже обученный StandardScaler (без утечки).
scaler=StandardScaler(with_mean=True,with_std=True)
scaler.fit(development[features].astype(float))
def scaled(frame):
    return pd.DataFrame(scaler.transform(frame[features].astype(float)),
                        index=frame.index,columns=features)
Z_train,Z_val,Z_dev,Z_test=[scaled(frame) for frame in (train,val,development,test)]
assert np.allclose(Z_dev.mean().to_numpy(),0,atol=1e-12)
assert np.allclose(Z_dev.std(ddof=0).to_numpy(),1,atol=1e-12)

# Уникальные точки плана: одинаковые 8 факторов -> одна строка X, несколько y и среднее.
# Группировка по исходным факторам исключает ложные совпадения из-за округления z.
groups = development.groupby(raw_features, sort=True, dropna=False)
group_ids = groups.ngroup().rename('group_id')
work = development.assign(group_id=group_ids)
summary = work.groupby('group_id').agg(y_mean=('ln_cpu_mark','mean'), n=('ln_cpu_mark','size'))
first_rows = work.groupby('group_id').head(1).set_index('group_id').loc[summary.index]
Z_group = scaled(first_rows)
X = sm.add_constant(Z_group,has_constant='add')
Y = summary.y_mean.astype(float)
weights = summary.n.astype(float)
# МНК по средним с весом кратности: Q = sum_i n_i (ybar_i - X_i B)^2.
# Это в точности эквивалентно МНК по всем 1365 наблюдениям.
model = sm.WLS(Y,X,weights=weights).fit()
Y_hat = model.fittedvalues
X_train=sm.add_constant(Z_train,has_constant='add')
X_val=sm.add_constant(Z_val,has_constant='add')
X_test=sm.add_constant(Z_test,has_constant='add')
X_dev_full=sm.add_constant(Z_dev,has_constant='add')
Y_hat_train=model.predict(X_train)
Y_hat_val=model.predict(X_val)
Y_hat_test=model.predict(X_test)
Y_hat_full=model.predict(X_dev_full)
resid_test=test.ln_cpu_mark-Y_hat_test
# Duan по всем наблюдениям (включая повторные точки), а не только по средним.
smearing=float(np.mean(np.exp(development.ln_cpu_mark-Y_hat_full)))
pred_cpu_test=np.exp(Y_hat_test)*smearing
assert X.shape[0]==len(summary)
assert summary.n.sum()==1365
assert len(summary)==1164
# Отдельно показываем невзвешенный МНК по средним (другая задача).
model_equal_points = sm.OLS(Y,X).fit()
print('Точек плана:',len(summary),'повторяющихся:',int((weights>1).sum()))
print('МНК по средним: основной — взвешенный по кратности; дополнительный — без весов.')
print('R² невзвешенного МНК по средним:',round(model_equal_points.rsquared,6))
scaler_table=pd.DataFrame({
    'Номер':[f'x{j}' for j in range(1,9)],
    'Фактор':feature_labels,
    'Признак до стандартизации':features,
    'Среднее на development':scaler.mean_,
    'Стандартное отклонение на development':scaler.scale_})
scaler_table.to_csv(TAB/'01_scaler_parameters.csv',index=False)
print('Размер обучающей матрицы X:',X.shape)
print('Возраст центрирован относительно:',f'{age_center:.8f}')
print('СТАНДАРТИЗАЦИЯ ВХОДИТ В ОСНОВНУЮ МОДЕЛЬ, y НЕ СТАНДАРТИЗОВАН.')
display(scaler_table.round(6))

Точек плана: 1164 повторяющихся: 161
МНК по средним: основной — взвешенный по кратности; дополнительный — без весов.
R² невзвешенного МНК по средним: 0.882305
Размер обучающей матрицы X: (1164, 9)
Возраст центрирован относительно: 13.87179487
СТАНДАРТИЗАЦИЯ ВХОДИТ В ОСНОВНУЮ МОДЕЛЬ, y НЕ СТАНДАРТИЗОВАН.


,Номер,Фактор,Признак до стандартизации,Среднее на development,Стандартное отклонение на development
0,x1,Число ядер,log1p_cores,1.226991,0.386781
1,x2,Базовая частота,base_clock_ghz,2.316484,0.711166
2,x3,TDP,log1p_tdp,3.544703,0.805385
3,x4,Кэш L3,log1p_l3,1.292567,0.765206
4,x5,Возраст,age_centered,-0.000000,2.889486
5,x6,Intel (относительно AMD),manufacturer_Intel,0.648352,0.477485
6,x7,Other (относительно AMD),manufacturer_Other,0.057143,0.232115
7,x8,Laptop (относительно Desktop),category_Laptop,0.536996,0.498629


## 2. Число уровней и повторы наблюдений

Ниже число **различных наблюдаемых значений каждого фактора** (это отдельная характеристика). Собственно **матрица уровней** в смысле лекции 2 — все строки плана $i$, значения $x_0=1$, $x_1,\ldots,x_8$ — экспортируется в Excel на следующем этапе. Данные наблюдательные: совпадения характеристик разных CPU не признаются повторными измерениями одного CPU.

In [4]:
labels = {
    'cores':'Количество ядер','base_clock_ghz':'Базовая частота, ГГц',
    'tdp_w':'TDP, Вт','l3_cache_mb':'L3, МБ','years_ago':'Возраст, лет',
}
levels = pd.DataFrame([
    {'Фактор':label,'Число уровней, final5':raw[col].nunique(),
     'Число уровней, development':development[col].nunique()}
    for col,label in labels.items()])
levels.loc[len(levels)] = [
    'Производитель (AMD / Intel / Other)',
    raw[['manufacturer_Intel','manufacturer_Other']].drop_duplicates().shape[0],
    development[['manufacturer_Intel','manufacturer_Other']].drop_duplicates().shape[0],
]
levels.loc[len(levels)] = [
    'Категория (Desktop / Laptop)',
    raw.category_Laptop.nunique(), development.category_Laptop.nunique(),
]
repeats = development.groupby(raw_features, dropna=False).size()
repeat_summary = pd.DataFrame([{
    'Всего наблюдений':len(development),
    'Уникальных точек плана':len(repeats),
    'Точек с повторами':int((repeats>1).sum()),
    'Максимальная кратность':int(repeats.max()),
    'Тип дублирования':'Неравномерное',
}])
levels.to_csv(TAB/'01_levels.csv',index=False)
repeat_summary.to_csv(TAB/'01b_repeated_points.csv',index=False)
display(levels)
display(repeat_summary)

,Фактор,"Число уровней, final5","Число уровней, development"
0,Количество ядер,12,12
1,"Базовая частота, ГГц",37,37
2,"TDP, Вт",128,117
3,"L3, МБ",25,24
4,"Возраст, лет",10,10
5,Производитель (AMD / Intel / Other),3,3
6,Категория (Desktop / Laptop),2,2


,Всего наблюдений,Уникальных точек плана,Точек с повторами,Максимальная кратность,Тип дублирования
0,1365,1164,161,6,Неравномерное


## 3. Уникальные точки плана и средние результаты

Матрица уровней содержит **одну строку на уникальное сочетание факторов**. В матрице результатов для этой строки записаны все наблюдения `yᵢ₁ ... yᵢ₆`, их число `nᵢ` и среднее `ȳᵢ`. Основной МНК выполняется по средним **с весами nᵢ**: `XᵀWX B = XᵀWȲ`, что сохраняет коэффициенты модели на 1365 наблюдениях. Дополнительно рассчитывается невзвешенный МНК по средним, который не эквивалентен исходному. Повторные точки могут соответствовать разным CPU, а не многократным измерениям одного CPU.

In [5]:
# Одна строка X на одну уникальную точку плана.
name_x=['x₀','x₁','x₂','x₃','x₄','x₅','x₆','x₇','x₈']
X_design=pd.DataFrame(X.to_numpy(),columns=name_x)
X_design.insert(0,'i',np.arange(1,len(X_design)+1))
# y_i1,...,y_i6 — реальные наблюдения для данной точки; пусто, если их меньше.
ys = work.groupby('group_id')['ln_cpu_mark'].apply(list).reindex(summary.index)
max_reps=int(weights.max())
rows=[]
for pos,(gid,values) in enumerate(ys.items(),start=1):
    values=list(values)
    rows.append([pos]+values+[None]*(max_reps-len(values))+[len(values),float(np.mean(values))])
Y_results=pd.DataFrame(rows,columns=['i']+[f'yᵢ{j}' for j in range(1,max_reps+1)]+['nᵢ','ȳᵢ (Y)'])
# Нормальные уравнения взвешенного МНК по средним: Xᵀ W X B = Xᵀ W Ȳ.
A=X.T @ X.mul(weights,axis=0)
rhs=X.T @ (weights*Y)
A.index=name_x;A.columns=name_x;rhs.index=name_x
assert np.linalg.matrix_rank(X.to_numpy())==9
X_design.to_csv(TAB/'02_matrix_levels.csv',index=False)
Y_results.to_csv(TAB/'03_matrix_results.csv',index=False)
A.rename_axis('factor').reset_index().to_csv(TAB/'04_XtWX.csv',index=False)
rhs.rename('XᵀWȲ').rename_axis('factor').reset_index().to_csv(TAB/'05_XtWY.csv',index=False)
# Альтернативный невзвешенный МНК по средним: XᵀX и XᵀȲ.
A_equal=X.T@X
rhs_equal=X.T@Y
# Сетка, белый фон, номер опыта i, единичный x0 и факторы x1 ... x8.
# Полностью локальный экспорт через openpyxl (без artifact_tool).
def build_xlsx(filename, sheet_name, headers, body, widths=None, legend=None, integer_cols=()):
    book = Workbook()
    sheet = book.active
    sheet.title = sheet_name
    nrows = len(body) + 1
    ncols = len(headers)
    assert not body or all(len(row) == ncols for row in body)

    # Оформление соответствует табличной схеме лекции 2:
    # первая строка — номера i, x0 ... x8 либо i, y_i1, среднее; белый фон,
    # чёрные буквы, тонкая сетка, выравнивание по центру.
    thin = Side(style='thin', color='555555')
    frame = Border(left=thin, right=thin, top=thin, bottom=thin)
    main_align = Alignment(horizontal='center', vertical='center')
    head_font = Font(name='Calibri', size=11, bold=True, color='000000')
    body_font = Font(name='Calibri', size=11, color='000000')
    white = PatternFill(fill_type='solid', fgColor='FFFFFF')
    whole_rows = [list(headers)] + list(body)

    for r, row in enumerate(whole_rows, start=1):
        for c, val in enumerate(row, start=1):
            # openpyxl сохраняет нативные числовые значения, а не строки.
            if isinstance(val, np.generic):
                val = val.item()
            cell = sheet.cell(row=r, column=c, value=val)
            cell.fill = white
            cell.border = frame
            cell.alignment = main_align
            cell.font = head_font if r == 1 else body_font
            if r > 1 and isinstance(val, (int, float)):
                cell.number_format = '0' if (c == 1 or c - 1 in integer_cols) else '0.000000'
        sheet.row_dimensions[r].height = 26 if r == 1 else 19
    for c, width in enumerate(widths or [16]*ncols, start=1):
        sheet.column_dimensions[get_column_letter(c)].width = width
    sheet.freeze_panes = 'A2'
    sheet.sheet_view.showGridLines = False
    sheet.print_options.horizontalCentered = True
    sheet.sheet_properties.pageSetUpPr.fitToPage = True
    sheet.page_setup.fitToWidth = 1
    sheet.print_title_rows = '1:1'

    # Дополнительный лист с расшифровкой обозначений не меняет
    # расположение матрицы на основном листе (лист открывается первым).
    if legend:
        info = book.create_sheet('Обозначения')
        for r, vals in enumerate(legend, start=1):
            for c, val in enumerate(vals, start=1):
                item = info.cell(row=r, column=c, value=val)
                item.font = head_font if r == 1 else body_font
                item.fill = white
                item.border = frame
                item.alignment = Alignment(vertical='center', horizontal='left', wrap_text=True)
            info.row_dimensions[r].height = 29 if r > 1 else 24
        info.column_dimensions['A'].width = 23
        info.column_dimensions['B'].width = 70
        info.freeze_panes = 'A2'
        info.sheet_view.showGridLines = False

    location = EXCEL / filename
    book.save(location)
    book.close()
    assert location.exists() and location.stat().st_size > 1000
    print('Excel:', location.name, '[строк:', nrows, ']')
    return location

legend=[['Обозначение','Определение'],['i','Номер уникальной точки плана от 1 до 1164'],
        ['x₀','Константный столбец из единиц']]
legend += [[f'x{j}',feature_labels[j-1]+' — z('+features[j-1]+')'] for j in range(1,9)]
legend += [['z(x)','(x - среднее по development) / стандартное отклонение по development'],
           ['Ȳ','Вектор средних значений ln(CPU Mark) в каждой точке плана'],
           ['Внимание','Совпадения характеристик разных CPU — повторные точки плана, не повторные измерения одного CPU']]

excel_files=[]
excel_files.append(build_xlsx('01_Матрица_уровней.xlsx','Матрица уровней',
    list(X_design.columns), X_design.astype(object).values.tolist(),
    [11]+[16]*9,legend,integer_cols=(1,)))
excel_files.append(build_xlsx('02_Матрица_результатов.xlsx','Матрица результатов',
    list(Y_results.columns),Y_results.astype(object).values.tolist(),
    [11]+[18]*max_reps+[12,22],[['Обозначение','Определение'],
               ['yᵢⱼ','Наблюдаемое ln(CPU Mark) для j-го CPU с одинаковыми факторами'],
               ['nᵢ','Число наблюдений в точке плана'],
               ['ȳᵢ (Y)','Среднее наблюдаемых ln(CPU Mark) в точке плана'],
               ['i','Номер уникальной точки плана']]))
ax_rows=[[name_x[j]]+[float(v) for v in A.iloc[j].to_numpy()] for j in range(9)]
excel_files.append(build_xlsx('03_XtWX_информационная_матрица.xlsx','XtX',
    [' ']+name_x,ax_rows,[11]+[17]*9,
    [['Обозначение','Определение'],['XᵀWX','Информационная матрица взвешенного МНК по средним, 9×9'],
     ['x₀','Свободный член'],['x₁…x₈','Стандартизованные регрессоры']]))
excel_files.append(build_xlsx('04_XtWY_правая_часть.xlsx','XtY',
    ['Фактор','XᵀWȲ'],[[name_x[j],float(rhs.iloc[j])] for j in range(9)],
    [17,22],[['Обозначение','Определение'],['XᵀWȲ','Правая часть взвешенных СНУ, 9 значений'],
             ['Y','Фактический ln(CPU Mark), а не прогноз']]))

print('\nПервые 5 строк матрицы уровней:')
display(X_design.head())
print('Первые 5 строк матрицы результатов:')
display(Y_results.head())
print('XᵀWX:')
display(A.round(4))
print('XᵀWȲ:')
display(rhs.to_frame('XᵀY').round(4))

Excel: 01_Матрица_уровней.xlsx [строк: 1165 ]
Excel: 02_Матрица_результатов.xlsx [строк: 1165 ]
Excel: 03_XtWX_информационная_матрица.xlsx [строк: 10 ]
Excel: 04_XtWY_правая_часть.xlsx [строк: 10 ]

Первые 5 строк матрицы уровней:


,i,x₀,x₁,x₂,x₃,x₄,x₅,x₆,x₇,x₈
0,1,1.000000,-1.380224,-2.413617,-1.819330,-1.159298,1.082616,0.736460,-0.246183,0.928553
1,2,1.000000,-1.380224,-2.132389,-1.819330,-1.689175,0.390452,0.736460,-0.246183,0.928553
2,3,1.000000,-1.380224,-1.991775,-1.819330,-1.159298,1.082616,0.736460,-0.246183,0.928553
3,4,1.000000,-1.380224,-1.991775,-0.650966,-1.689175,1.082616,-1.357848,4.062019,0.928553
4,5,1.000000,-1.380224,-1.991775,-0.615147,-1.689175,1.082616,-1.357848,4.062019,0.928553


Первые 5 строк матрицы результатов:


,i,yᵢ1,yᵢ2,yᵢ3,yᵢ4,yᵢ5,yᵢ6,nᵢ,ȳᵢ (Y)
0,1,4.584967,NaN,NaN,NaN,NaN,NaN,1,4.584967
1,2,4.787492,NaN,NaN,NaN,NaN,NaN,1,4.787492
2,3,4.948760,NaN,NaN,NaN,NaN,NaN,1,4.948760
3,4,4.927254,NaN,NaN,NaN,NaN,NaN,1,4.927254
4,5,4.867534,NaN,NaN,NaN,NaN,NaN,1,4.867534


XᵀWX:


,x₀,x₁,x₂,x₃,x₄,x₅,x₆,x₇,x₈
x₀,1365.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000
x₁,0.000000,1365.000000,555.686500,511.766700,1086.115500,-641.912600,66.345900,-410.610000,-359.131900
x₂,-0.000000,555.686500,1365.000000,988.757500,643.129700,-280.409700,32.429400,-209.518100,-757.969600
x₃,-0.000000,511.766700,988.757500,1365.000000,502.365300,187.329100,-265.493200,-62.546000,-809.404800
x₄,0.000000,1086.115500,643.129700,502.365300,1365.000000,-622.678800,121.398200,-567.629900,-277.258300
x₅,0.000000,-641.912600,-280.409700,187.329100,-622.678800,1365.000000,-411.353400,263.905700,-15.287300
x₆,0.000000,66.345900,32.429400,-265.493200,121.398200,-411.353400,1365.000000,-456.290800,166.989700
x₇,0.000000,-410.610000,-209.518100,-62.546000,-567.629900,263.905700,-456.290800,1365.000000,208.349600
x₈,-0.000000,-359.131900,-757.969600,-809.404800,-277.258300,-15.287300,166.989700,208.349600,1365.000000


XᵀWȲ:


,XᵀY
x₀,9790.864000
x₁,1329.875300
x₂,1012.732200
x₃,669.145700
x₄,1297.896800
x₅,-1023.547000
x₆,404.943400
x₇,-589.394500
x₈,-432.980100


In [18]:
X_design.drop_duplicates().shape == X_design.shape

True

## 4. СНУ взвешенного МНК по средним

Решается `XᵀWX B = XᵀWȲ`, где `W=diag(nᵢ)`. Здесь X содержит 1164 уникальные точки, а Y — средние результаты. Для сравнения показан невзвешенный МНК по средним.

In [6]:
B=pd.Series(np.linalg.solve(A.to_numpy(),rhs.to_numpy()),
            index=X.columns,name='b_SNU')
comparison=pd.DataFrame({'Фактор':['const']+features,
                         'b_SNU':B.to_numpy(),
                         'b_WLS':model.params.to_numpy()})
comparison['abs_difference']=(comparison.b_SNU-comparison.b_WLS).abs()
np.testing.assert_allclose(B.to_numpy(),model.params.to_numpy(),rtol=1e-9,atol=1e-9)
comparison.to_csv(TAB/'06_SNU_vs_OLS.csv',index=False)
print('Наибольшее расхождение СНУ vs OLS:',comparison.abs_difference.max())
print('Уравнение ŷ для z-факторов:')
terms=[f'{B.iloc[0]:+.6f}']
for i, b in enumerate(B.iloc[1:],1):
    terms.append(f'{b:+.6f}·z{i}')
print('ŷ = '+' '.join(terms))
display(comparison.round(10))


Наибольшее расхождение СНУ vs OLS: 9.992007221626409e-15
Уравнение ŷ для z-факторов:
ŷ = +7.172794 +0.483894·z1 +0.346360·z2 +0.108749·z3 +0.203161·z4 -0.319625·z5 +0.144079·z6 -0.048037·z7 +0.094318·z8


,Фактор,b_SNU,b_WLS,abs_difference
0,const,7.172794,7.172794,0.000000
1,log1p_cores,0.483894,0.483894,0.000000
2,base_clock_ghz,0.346360,0.346360,0.000000
3,log1p_tdp,0.108749,0.108749,0.000000
4,log1p_l3,0.203161,0.203161,0.000000
5,age_centered,-0.319625,-0.319625,0.000000
6,manufacturer_Intel,0.144079,0.144079,0.000000
7,manufacturer_Other,-0.048037,-0.048037,0.000000
8,category_Laptop,0.094318,0.094318,0.000000


## 5. Стандартные ошибки, важность факторов и интерпретации

Для сравнения всех восьми факторов применяется величина $|b_j|$ при увеличении стандартизованного регрессора на **одну сигму**. Dummy-переменные также стандартизированы, поэтому их $|b_j|$ зависят от долей категорий и **не равны** величине эффекта полного переключения `0 → 1`.

Для интерпретации в исходных единицах коэффициент $b_j$ переводится обратно: $b_j^{\mathrm{raw}}=b_j/s_j$. Это **не другая обученная модель**, а алгебраически эквивалентная запись той же стандартизованной регрессии. Для линейного по исходной величине признака изменение на $\Delta$ связано с изменением прогноза CPU Mark на $100(e^{b_j^{raw}\Delta}-1)\%$ при прочих равных. Для логарифмированных характеристик используется точная разность логарифмов.

Важность дополнительно проверяется через $\Delta R_j^2=R^2_{full}-R^2_{without\ j}$ с переоцениванием модели по оставшимся стандартизованным признакам.

In [7]:
coefficients=pd.DataFrame({'Фактор':['Свободный член']+feature_labels,
        'Регрессор':['const']+features,
        'b_стандартизированный':model.params.to_numpy(),
        'SE':model.bse.to_numpy(),
        't':model.tvalues.to_numpy(),
        'p':model.pvalues.to_numpy()})
coefficients.to_csv(TAB/'07_coefficients_SE_t_p.csv',index=False)

# Аналитический возврат к физическим единицам для интерпретации:
original_beta=pd.Series(model.params.iloc[1:].to_numpy()/scaler.scale_,index=features)
original_intercept=float(model.params['const']-(original_beta.to_numpy()*scaler.mean_).sum())
# Это не повторное оценивание: прогнозы обязаны точно совпадать.
original_prediction=original_intercept+development[features].to_numpy()@original_beta.to_numpy()
np.testing.assert_allclose(original_prediction,Y_hat_full.to_numpy(),atol=1e-10)
original_coef=pd.DataFrame({'Фактор':feature_labels,'Признак':features,
                            'b_в_исходных_единицах':original_beta.to_numpy(),
                            'b_стандартизированный':model.params.iloc[1:].to_numpy()})
original_coef.to_csv(TAB/'08_interpretation_coefficients.csv',index=False)

importance=pd.DataFrame({'Фактор':feature_labels,'Признак':features,
    'b_при_росте_X_на_1_SD':model.params.iloc[1:].to_numpy(),
    'abs_b':np.abs(model.params.iloc[1:].to_numpy())})
importance=importance.sort_values('abs_b',ascending=False).reset_index(drop=True)
importance.insert(0,'Место',np.arange(1,len(importance)+1))
importance.to_csv(TAB/'09_importance_standardized.csv',index=False)

# Падение R² на development при исключении регрессора и оценка на test.
drop_results=[]
for f,label in zip(features,feature_labels):
    cols=[c for c in features if c!=f]
    submodel=sm.WLS(Y,sm.add_constant(Z_group[cols],has_constant='add'),weights=weights).fit()
    sub_test=submodel.predict(sm.add_constant(Z_test[cols],has_constant='add'))
    drop_results.append({'Фактор':label,'Регрессор':f,
      'R²_development_без':submodel.rsquared,
      'Падение_R²_development':model.rsquared-submodel.rsquared,
      'R²_test_без':r2_score(test.ln_cpu_mark,sub_test),
      'Падение_R²_test':r2_score(test.ln_cpu_mark,Y_hat_test)-r2_score(test.ln_cpu_mark,sub_test)})
importance_drop=pd.DataFrame(drop_results).sort_values('Падение_R²_development',ascending=False)
importance_drop.to_csv(TAB/'10_importance_delta_R2.csv',index=False)
# Производитель — категориальный фактор с двумя dummy-признаками.
cols_without_manufacturer=[f for f in features if f not in ['manufacturer_Intel','manufacturer_Other']]
manufacturer_model=sm.WLS(Y,sm.add_constant(Z_group[cols_without_manufacturer],has_constant='add'),weights=weights).fit()
manufacturer_delta=float(model.rsquared-manufacturer_model.rsquared)

# Интерпретация +1 год, +1 ГГц и типичных изменений логарифмированных X.
def percent_change(delta_log): return 100*np.expm1(delta_log)
interpretations=[
    ('Возраст +1 год',percent_change(original_beta['age_centered']),
     'при прочих равных, CPU Mark ожидается изменить на указанную долю'),
    ('Базовая частота +1 ГГц',percent_change(original_beta['base_clock_ghz']),
     'при прочих равных, рост базовой частоты на 1 ГГц'),
    ('Ядра 4 → 5',percent_change(original_beta['log1p_cores']*np.log(6/5)),
     'изменение числа ядер с 4 до 5'),
    ('TDP 65 → 75 Вт',percent_change(original_beta['log1p_tdp']*np.log(76/66)),
     'изменение TDP с 65 до 75 Вт'),
    ('L3 8 → 9 МБ',percent_change(original_beta['log1p_l3']*np.log(10/9)),
     'увеличение L3 с 8 до 9 МБ'),
    ('Intel вместо AMD',percent_change(original_beta['manufacturer_Intel']),
     'бинарный переход Intel=0 → 1 при Other=0'),
    ('Other вместо AMD',percent_change(original_beta['manufacturer_Other']),
     'бинарный переход Other=0 → 1 при Intel=0'),
    ('Laptop вместо Desktop',percent_change(original_beta['category_Laptop']),
     'бинарный переход Laptop=0 → 1'),
]
interpretations_df=pd.DataFrame(interpretations,columns=['Изменение фактора','Ожидаемое изменение CPU Mark, %','Условие'])
interpretations_df.to_csv(TAB/'11_interpretations_percent.csv',index=False)

print('\nОценки коэффициентов и их значимость:')
display(coefficients.round(6))
print('\nВажность по |b| после стандартизации (все 8, включая dummy):')
display(importance.round(6))
print('\nУникальный вклад регрессоров через ΔR²:')
display(importance_drop.round(6))
print(f'Производитель как единый фактор: снижение R² при удалении обеих dummy = {manufacturer_delta:.6f}')
print('\nИнтерпретации в исходных единицах (без повторного обучения модели):')
display(interpretations_df.round(3))
print('В частности, возраст +1 год: '+f'{interpretations_df.iloc[0,1]:+.2f}% CPU Mark при прочих равных.')



Оценки коэффициентов и их значимость:


,Фактор,Регрессор,b_стандартизированный,SE,t,p
0,Свободный член,const,7.172794,0.011771,609.369961,0.000000
1,Число ядер,log1p_cores,0.483894,0.020872,23.184140,0.000000
2,Базовая частота,base_clock_ghz,0.346360,0.020194,17.151518,0.000000
3,TDP,log1p_tdp,0.108749,0.021666,5.019324,0.000001
4,Кэш L3,log1p_l3,0.203161,0.021989,9.239129,0.000000
5,Возраст,age_centered,-0.319625,0.016320,-19.585277,0.000000
6,Intel (относительно AMD),manufacturer_Intel,0.144079,0.013470,10.696242,0.000000
7,Other (относительно AMD),manufacturer_Other,-0.048037,0.014160,-3.392349,0.000716
8,Laptop (относительно Desktop),category_Laptop,0.094318,0.015594,6.048356,0.000000



Важность по |b| после стандартизации (все 8, включая dummy):


,Место,Фактор,Признак,b_при_росте_X_на_1_SD,abs_b
0,1,Число ядер,log1p_cores,0.483894,0.483894
1,2,Базовая частота,base_clock_ghz,0.346360,0.346360
2,3,Возраст,age_centered,-0.319625,0.319625
3,4,Кэш L3,log1p_l3,0.203161,0.203161
4,5,Intel (относительно AMD),manufacturer_Intel,0.144079,0.144079
5,6,TDP,log1p_tdp,0.108749,0.108749
6,7,Laptop (относительно Desktop),category_Laptop,0.094318,0.094318
7,8,Other (относительно AMD),manufacturer_Other,-0.048037,0.048037



Уникальный вклад регрессоров через ΔR²:


,Фактор,Регрессор,R²_development_без,Падение_R²_development,R²_test_без,Падение_R²_test
0,Число ядер,log1p_cores,0.833471,0.052886,0.812132,0.061603
4,Возраст,age_centered,0.848615,0.037742,0.848462,0.025272
1,Базовая частота,base_clock_ghz,0.857413,0.028944,0.827489,0.046246
5,Intel (относительно AMD),manufacturer_Intel,0.875100,0.011257,0.866773,0.006962
3,Кэш L3,log1p_l3,0.877958,0.008399,0.868805,0.004929
7,Laptop (относительно Desktop),category_Laptop,0.882758,0.003599,0.869389,0.004345
2,TDP,log1p_tdp,0.883878,0.002479,0.874037,-0.000302
6,Other (относительно AMD),manufacturer_Other,0.885225,0.001132,0.875651,-0.001916


Производитель как единый фактор: снижение R² при удалении обеих dummy = 0.016759

Интерпретации в исходных единицах (без повторного обучения модели):


,Изменение фактора,"Ожидаемое изменение CPU Mark, %",Условие
0,Возраст +1 год,-10.472000,"при прочих равных, CPU Mark ожидается изменить..."
1,Базовая частота +1 ГГц,62.748000,"при прочих равных, рост базовой частоты на 1 ГГц"
2,Ядра 4 → 5,25.621000,изменение числа ядер с 4 до 5
3,TDP 65 → 75 Вт,1.923000,изменение TDP с 65 до 75 Вт
4,L3 8 → 9 МБ,2.837000,увеличение L3 с 8 до 9 МБ
5,Intel вместо AMD,35.222000,бинарный переход Intel=0 → 1 при Other=0
6,Other вместо AMD,-18.694000,бинарный переход Other=0 → 1 при Intel=0
7,Laptop вместо Desktop,20.823000,бинарный переход Laptop=0 → 1


В частности, возраст +1 год: -10.47% CPU Mark при прочих равных.


## 6. Обучающая выборка против тестовой: $R^2$

Итоговая модель оценена на `development = train + validation` (1365 CPU). `train` (1092) — часть этой обучающей выборки, так что её метрики после финального оценивания тоже **внутривыборочные**; тест из 300 CPU никак не участвовал в оценивании весов или стандартизатора. Основное сравнение — `development` и `test`, дополнительно приводится значение на `train`.

Если тестовый $R^2$ близок к обучающему, это свидетельствует, что на данной отложенной выборке заметного ухудшения качества не наблюдается; но это не доказывает отсутствие всех форм переобучения.

In [8]:
r2_train=float(r2_score(train.ln_cpu_mark,Y_hat_train))
r2_val=float(r2_score(val.ln_cpu_mark,Y_hat_val))
r2_dev=float(r2_score(development.ln_cpu_mark,Y_hat_full))
r2_test=float(r2_score(test.ln_cpu_mark,Y_hat_test))
gap_dev_test=r2_dev-r2_test
metrics=pd.DataFrame([
  {'Выборка':'Train (часть development)','n':len(train),'R²':r2_train,
   'RMSE(log)':np.sqrt(mean_squared_error(train.ln_cpu_mark,Y_hat_train)),
   'MAE(log)':mean_absolute_error(train.ln_cpu_mark,Y_hat_train)},
  {'Выборка':'Validation (часть development)','n':len(val),'R²':r2_val,
   'RMSE(log)':np.sqrt(mean_squared_error(val.ln_cpu_mark,Y_hat_val)),
   'MAE(log)':mean_absolute_error(val.ln_cpu_mark,Y_hat_val)},
  {'Выборка':'Development (обучение модели)','n':len(development),'R²':r2_dev,
   'RMSE(log)':np.sqrt(mean_squared_error(development.ln_cpu_mark,Y_hat_full)),
   'MAE(log)':mean_absolute_error(development.ln_cpu_mark,Y_hat_full)},
  {'Выборка':'Test (отложенная)','n':len(test),'R²':r2_test,
   'RMSE(log)':np.sqrt(mean_squared_error(test.ln_cpu_mark,Y_hat_test)),
   'MAE(log)':mean_absolute_error(test.ln_cpu_mark,Y_hat_test)},
])
metrics.to_csv(TAB/'12_train_validation_development_test_R2.csv',index=False)
metric_cpu=pd.DataFrame([{'Масштаб':'CPU Mark с поправкой Duan','n_test':len(test),
    'R²':r2_score(test.cpu_mark,pred_cpu_test),
    'RMSE':np.sqrt(mean_squared_error(test.cpu_mark,pred_cpu_test)),
    'MAE':mean_absolute_error(test.cpu_mark,pred_cpu_test),'Duan':smearing}])
metric_cpu.to_csv(TAB/'13_test_CPU_Mark_original_scale.csv',index=False)
quality=pd.DataFrame([{'n_dev':len(development),'R²_dev':r2_dev,
    'Adj_R²_dev':model.rsquared_adj,'F':model.fvalue,'p(F)':model.f_pvalue,
    'R²_train':r2_train,'R²_test':r2_test,'R²_gap_dev_test':gap_dev_test}])
quality.to_csv(TAB/'14_quality_summary.csv',index=False)
display(metrics.round(6))
print('CPU Mark в исходных баллах:')
display(metric_cpu.round(6))
print(f'Разница R²(development) − R²(test) = {gap_dev_test:.6f} ({100*gap_dev_test:.2f} п.п.)')
print('Вывод: '+('R² на тесте близок к обучающему; выраженного провала на независимой выборке нет.'
    if abs(gap_dev_test)<.03 else 'R² заметно различаются; требуется более подробный анализ обобщающей способности.'))
print('Значения на train/validation после окончательного обучения на development не являются независимой оценкой.')


,Выборка,n,R²,RMSE(log),MAE(log)
0,Train (часть development),1092,0.884254,0.401959,0.302736
1,Validation (часть development),273,0.884527,0.412483,0.305589
2,Development (обучение модели),1365,0.884312,0.404086,0.303307
3,Test (отложенная),300,0.873735,0.433776,0.308027


CPU Mark в исходных баллах:


,Масштаб,n_test,R²,RMSE,MAE,Duan
0,CPU Mark с поправкой Duan,300,0.677667,1519.781781,623.176662,1.080130


Разница R²(development) − R²(test) = 0.010577 (1.06 п.п.)
Вывод: R² на тесте близок к обучающему; выраженного провала на независимой выборке нет.
Значения на train/validation после окончательного обучения на development не являются независимой оценкой.


## 7. Графики для актуального отчёта (PNG; без анализа остатков)

In [9]:
def save(fig, filename):
    path = FIG/filename
    fig.savefig(path,dpi=200,bbox_inches='tight',facecolor='white')
    if DISPLAY_FIGURES:
        display(fig)
    plt.close(fig)
    print('Сохранено:',path.name)

# 01 — CPU Mark в исходном масштабе относительно числовых факторов
num_raw=['cores','base_clock_ghz','tdp_w','l3_cache_mb','years_ago']
fig,axes=plt.subplots(2,3,figsize=(12.6,7.0),constrained_layout=True)
for ax,col in zip(axes.flat,num_raw):
    xx=train[col].to_numpy(float); yy=train.cpu_mark.to_numpy(float)
    ax.scatter(xx,yy,s=9,alpha=.35)
    slope,shift=np.polyfit(xx,yy,1)
    xp=np.linspace(xx.min(),xx.max(),60)
    ax.plot(xp,slope*xp+shift,color='firebrick',lw=1.4)
    ax.set_title(f'CPU Mark от {col} (r={np.corrcoef(xx,yy)[0,1]:.2f})')
    ax.set_xlabel(col);ax.set_ylabel('CPU Mark');ax.grid(alpha=.15)
axes.flat[-1].axis('off')
save(fig,'01_initial_numeric.png')

# 02 — категории производителя и тип CPU
fig,axes=plt.subplots(1,2,figsize=(12.6,4.1),constrained_layout=True)
maker=np.select([train.manufacturer_Intel.eq(1),train.manufacturer_Other.eq(1)],
                ['Intel','Other'],default='AMD')
for ax,cat,order,title in [
    (axes[0],maker,['AMD','Intel','Other'],'Производитель'),
    (axes[1],np.where(train.category_Laptop.eq(1),'Laptop','Desktop'),
     ['Desktop','Laptop'],'Категория')
]:
    values=[train.cpu_mark.to_numpy()[cat==label] for label in order]
    ax.boxplot(values,tick_labels=order,showfliers=False)
    ax.set_title(title);ax.set_ylabel('CPU Mark');ax.grid(axis='y',alpha=.15)
save(fig,'02_initial_categories.png')

Сохранено: 01_initial_numeric.png
Сохранено: 02_initial_categories.png


In [10]:
# 03 — связь с признаками после log1p, до стандартизации (визуальное обоснование преобразования)
view=features[:5]
fig,axes=plt.subplots(2,3,figsize=(12.6,7.0),constrained_layout=True)
for ax,col in zip(axes.flat,view):
    xx=train[col].to_numpy(float);yy=train.ln_cpu_mark.to_numpy(float)
    ax.scatter(xx,yy,s=9,alpha=.32)
    slope,shift=np.polyfit(xx,yy,1);xp=np.linspace(xx.min(),xx.max(),60)
    ax.plot(xp,slope*xp+shift,color='firebrick',lw=1.4)
    ax.set_title(f'ln(CPU Mark) от {col} (r={np.corrcoef(xx,yy)[0,1]:.2f})')
    ax.set_xlabel(col);ax.set_ylabel('ln(CPU Mark)');ax.grid(alpha=.15)
axes.flat[-1].axis('off')
save(fig,'03_transformed_numeric.png')

# 04 — графики регрессоров z, ИМЕННО ЭТИ факторы использует обученная модель.
fig,axes=plt.subplots(2,3,figsize=(12.6,7.0),constrained_layout=True)
for ax,col in zip(axes.flat,view):
    xx=Z_train[col].to_numpy();yy=train.ln_cpu_mark.to_numpy()
    ax.scatter(xx,yy,s=9,alpha=.32)
    slope,shift=np.polyfit(xx,yy,1);xp=np.linspace(xx.min(),xx.max(),60)
    ax.plot(xp,slope*xp+shift,color='firebrick',lw=1.4)
    ax.set_title('Обучающий регрессор: z('+col+')')
    ax.set_xlabel('z('+col+')');ax.set_ylabel('ln(CPU Mark)');ax.grid(alpha=.15)
axes.flat[-1].axis('off')
save(fig,'04_standardized_numeric.png')

Сохранено: 03_transformed_numeric.png
Сохранено: 04_standardized_numeric.png


In [11]:
# 08 — test: наблюдения vs прогноз и гистограмма ошибок прогноза на test
fig,axes=plt.subplots(1,2,figsize=(12.6,4.2),constrained_layout=True)
axes[0].scatter(test.ln_cpu_mark,Y_hat_test,s=16,alpha=.45)
lo=min(test.ln_cpu_mark.min(),Y_hat_test.min());hi=max(test.ln_cpu_mark.max(),Y_hat_test.max())
axes[0].plot([lo,hi],[lo,hi],color='firebrick',lw=1.5)
axes[0].set_title(f'Факт vs прогноз (test, R²={r2_score(test.ln_cpu_mark,Y_hat_test):.3f})')
axes[0].set_xlabel('Фактический ln(CPU Mark)');axes[0].set_ylabel('Прогнозный ln(CPU Mark)')
axes[1].hist(resid_test,bins=32)
axes[1].axvline(0,color='firebrick',lw=1.5)
axes[1].set_title('Распределение ошибок test')
axes[1].set_xlabel('Факт − прогноз (ln CPU Mark)');axes[1].set_ylabel('Частота')
for ax in axes:ax.grid(alpha=.15)
save(fig,'08_test_predictions.png')

# 09 — распределение логарифма целевой переменной с наложенной нормальной плотностью
# По всем 1665 наблюдениям final5.csv, как на рисунке в текущем отчёте.
log_y=np.log(raw.cpu_mark.astype(float))
mu=float(log_y.mean()); sigma=float(log_y.std(ddof=1))
fig,ax=plt.subplots(figsize=(9,6))
ax.hist(log_y,bins=35,density=True,alpha=.75,edgecolor='black')
xx=np.linspace(log_y.min()-.3,log_y.max()+.3,500)
ax.plot(xx,stats.norm.pdf(xx,loc=mu,scale=sigma),linewidth=2.5,color='darkorange')
ax.set_title('Распределение ln(CPU Mark)')
ax.set_xlabel('ln(CPU Mark)');ax.set_ylabel('Плотность');ax.grid(True,alpha=.25)
ax.text(.02,.96,f'n = {len(log_y)}\nμ = {mu:.3f}\nσ = {sigma:.3f}',
        transform=ax.transAxes,va='top',
        bbox=dict(boxstyle='round',facecolor='white',alpha=.9))
save(fig,'09_log_target_normal_distribution.png')

Сохранено: 08_test_predictions.png
Сохранено: 09_log_target_normal_distribution.png


## 8. Дополнительные графики: важность факторов и сравнение $R^2$

In [12]:
# 11 — абсолютные коэффициенты на стандартизованных X
fig,ax=plt.subplots(figsize=(10.4,5.7),constrained_layout=True)
order=importance.iloc[::-1]
ax.barh(order['Фактор'],order['abs_b'],color='#477AB2')
for i,v in enumerate(order['abs_b']): ax.text(v+.005,i,f'{v:.3f}',va='center',fontsize=9)
ax.set_xlabel('|b| при изменении стандартизованного X на 1 σ')
ax.set_title('Важность регрессоров по абсолютным стандартизованным коэффициентам')
ax.set_xlim(0,float(order['abs_b'].max())*1.2);ax.grid(axis='x',alpha=.15)
save(fig,'11_importance_standardized_coefficients.png')

# 12 — уникальный вклад: сколько R² потеряется без отдельного фактора.
fig,ax=plt.subplots(figsize=(10.4,5.7),constrained_layout=True)
order=importance_drop.iloc[::-1]
ax.barh(order['Фактор'],order['Падение_R²_development'],color='#477AB2')
for i,v in enumerate(order['Падение_R²_development']):
    ax.text(v+.0005,i,f'{v:.4f}',va='center',fontsize=9)
ax.set_title('Уникальная важность регрессоров: уменьшение R² при исключении')
ax.set_xlabel('R² полной модели − R² без регрессора')
ax.set_xlim(0,float(order['Падение_R²_development'].max())*1.2)
ax.grid(axis='x',alpha=.15)
save(fig,'12_importance_drop_R2.png')

# 13 — сравнение R² development/train/test.
fig,ax=plt.subplots(figsize=(7.6,4.7),constrained_layout=True)
vals=[r2_train,r2_dev,r2_test]
name=['Train (1092)','Обучение: development (1365)','Test (300)']
bars=ax.bar(name,vals,color=['#97B7D4','#477AB2','#708AA7'])
for b,v in zip(bars,vals):ax.text(b.get_x()+b.get_width()/2,v+.006,f'{v:.4f}',
                                   ha='center',va='bottom',fontsize=11)
ax.set_ylim(0,1.02);ax.set_ylabel('R², целевая переменная ln(CPU Mark)')
ax.set_title('Качество регрессии: обучающие и независимые наблюдения')
ax.tick_params(axis='x',labelrotation=12)
ax.grid(axis='y',alpha=.18)
save(fig,'13_train_development_test_R2.png')

Сохранено: 11_importance_standardized_coefficients.png
Сохранено: 12_importance_drop_R2.png
Сохранено: 13_train_development_test_R2.png


## 9. Приложение: девять уравнений СНУ

In [13]:
# x_i0=1. Каждое уравнение располагается на двух строках по 5 и 4 слагаемых.
def eq_lines(j):
    parts=[rf'a_{{{j}{m}}}b_{{{m}}}' for m in range(9)]
    return ('+'.join(parts[:5])+r'+', '+'.join(parts[5:])+rf'=c_{{{j}}}')
for j in range(9):
    first,second=eq_lines(j)
    fig,ax=plt.subplots(figsize=(13,1.55))
    ax.axis('off')
    ax.text(.01,.72,'$'+first+'$',fontsize=15,va='center')
    ax.text(.01,.23,'$'+second+'$',fontsize=15,va='center')
    fig.savefig(APP/f'SNU_equation_{j+1:02d}.png',dpi=190,bbox_inches='tight',pad_inches=.1)
    plt.close(fig)
fig,axes=plt.subplots(9,1,figsize=(12.6,14),constrained_layout=True)
for j,ax in enumerate(axes):
    a,bline=eq_lines(j)
    ax.axis('off')
    ax.text(.01,.74,'$'+a+'$',fontsize=14,va='center')
    ax.text(.01,.18,'$'+bline+'$',fontsize=14,va='center')
save(fig,'10_SNU_nine_equations.png')

Сохранено: 10_SNU_nine_equations.png


## 10. Контроль воспроизводимости, файлы Excel, архив

In [14]:
expected_figures=[
 '01_initial_numeric.png','02_initial_categories.png',
 '03_transformed_numeric.png','04_standardized_numeric.png',
 '08_test_predictions.png',
 '09_log_target_normal_distribution.png',
 '10_SNU_nine_equations.png',
 '11_importance_standardized_coefficients.png',
 '12_importance_drop_R2.png',
 '13_train_development_test_R2.png',
]
missing=[f for f in expected_figures if not (FIG/f).exists()]
assert not missing, f'Нет сохранённых графиков: {missing}'
assert len(list(APP.glob('SNU_equation_*.png')))==9
assert all(f.exists() for f in excel_files)
# Контроль: прогнозы той же модели, оценки коэффициентов только в другом масштабе.
assert abs(r2_dev-0.8843116042819963)<1e-8
assert abs(r2_test-0.873735)<1e-5
assert abs(original_beta['age_centered']-(-.1106162))<1e-4
assert abs(smearing-1.080130)<1e-5

notes=('Единый расчёт стандартизованной модели CPU Mark (8 регрессоров).\n'
       'Матрица уровней и матрица результатов в стиле лекции 2 (слайды 12–13).\n'
       'Одна точка плана = одна строка X; все фактические y_ij и их среднее показаны в Y.\n'
       'МНК по средним взвешен кратностями n_i, что сохраняет исходные коэффициенты.\n'
       'Все значения z получены по средним и стандартным отклонениям development.\n'
       'Без раздела анализа остатков модели; сохранена проверка прогнозов test.\n'
       'Никаких листингов кода для Word-отчёта.\n\n'
       'Графики:\n'+ '\n'.join(expected_figures)+'\n')
(RESULTS_DIR/'README.txt').write_text(notes,encoding='utf-8')
all_zip=RESULTS_DIR/'all_results.zip'
with zipfile.ZipFile(all_zip,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in expected_figures:
        path = FIG/filename
        archive.write(path,arcname=str(path.relative_to(RESULTS_DIR)))
    for folder in (TAB,EXCEL,APP):
        for path in sorted(folder.glob('*')):
            if path.is_file():archive.write(path,arcname=str(path.relative_to(RESULTS_DIR)))
    archive.write(RESULTS_DIR/'README.txt',arcname='README.txt')
print('ГОТОВО:')
print('  графиков PNG:',len(expected_figures))
print('  уравнений СНУ в PNG:',len(list(APP.glob('*.png'))))
print('  Excel-файлов (openpyxl):',len(excel_files))
print('  CSV-таблиц:',len(list(TAB.glob('*.csv'))))
print('  архив:',all_zip.resolve())
print('  модель обучена по СТАНДАРТИЗОВАННЫМ X, константа и остальные оценки соответствуют этому масштабу')
print('  R² dev/test =',round(r2_dev,6),'/',round(r2_test,6))


ГОТОВО:
  графиков PNG: 10
  уравнений СНУ в PNG: 9
  Excel-файлов (openpyxl): 4
  CSV-таблиц: 16
  архив: /home/monkofwst/codes/smmo/pravki_final/lab1_pumba_standardized_results/all_results.zip
  модель обучена по СТАНДАРТИЗОВАННЫМ X, константа и остальные оценки соответствуют этому масштабу
  R² dev/test = 0.884312 / 0.873735
